In [1]:
import numpy as np

In [12]:
import os

In [2]:
import pandas as pd

In [3]:
import matplotlib.pyplot as plt

In [84]:
from sklearn.preprocessing import MinMaxScaler

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [6]:
from tensorflow.keras.models import Sequential

In [7]:
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [8]:
from tensorflow.keras.callbacks import EarlyStopping

In [9]:
from tensorflow.keras.optimizers import Adam

In [24]:
data = pd.read_csv('../../Data_Collection_and_Understanding/processed_data.csv')

In [25]:
data.head()

,Home ID,Appliance Type,Energy Consumption (kWh),Outdoor Temperature (°C),Season,Household Size,timestamp
0,94,Fridge,0.20,-1.0,Fall,2,2023-12-02 21:12:00
1,435,Oven,0.23,31.1,Summer,5,2023-08-06 20:11:00
2,466,Dishwasher,0.32,21.3,Fall,3,2023-11-21 06:39:00
3,496,Heater,3.92,-4.2,Winter,1,2023-01-21 21:56:00
4,137,Microwave,0.44,34.5,Summer,5,2023-08-26 04:31:00


In [34]:
data['timestamp'] = pd.to_datetime(data['timestamp'])

In [39]:
final_data = data.sort_values(by ='timestamp')

In [44]:
data.head(2)

,Home ID,Appliance Type,Energy Consumption (kWh),Outdoor Temperature (°C),Season,Household Size,timestamp
0,94,Fridge,0.20,-1.0,Fall,2,2023-12-02 21:12:00
1,435,Oven,0.23,31.1,Summer,5,2023-08-06 20:11:00


In [47]:
final_data.head(5)

,Home ID,Appliance Type,Energy Consumption (kWh),Outdoor Temperature (°C),Season,Household Size,timestamp
51009,140,Lights,1.00,-5.9,Winter,1,2023-01-01 00:07:00
26231,298,Lights,1.09,22.0,Winter,1,2023-01-01 00:13:00
8601,469,Fridge,0.30,-1.2,Winter,3,2023-01-01 00:24:00
32992,398,Fridge,0.50,35.8,Winter,2,2023-01-01 00:26:00
92541,293,Washing Machine,1.12,2.7,Winter,3,2023-01-01 00:30:00


Here I sorted all the values in ascending order by timestamp, so that I can use it to train using LSTM

In [69]:
energy = final_data[['Energy Consumption (kWh)']]

In [70]:
energy.shape

(99049, 1)

In [71]:
energy.head()

,Energy Consumption (kWh)
51009,1.00
26231,1.09
8601,0.30
32992,0.50
92541,1.12


In [50]:
scaler = MinMaxScaler( feature_range = (0,1))

In [72]:
energy_scaled = scaler.fit_transform(energy)

In [73]:
energy_scaled.shape

(99049, 1)

In [108]:
energy_scaled[:30]

array([[0.18947368],
       [0.20842105],
       [0.04210526],
       [0.08421053],
       [0.21473684],
       [0.33263158],
       [0.12210526],
       [0.35578947],
       [0.90947368],
       [0.29894737],
       [0.35789474],
       [0.38526316],
       [0.28631579],
       [0.36421053],
       [0.28      ],
       [0.39578947],
       [0.11368421],
       [0.31578947],
       [0.89894737],
       [0.04631579],
       [0.15578947],
       [0.38105263],
       [0.01473684],
       [0.05473684],
       [0.66105263],
       [0.02315789],
       [0.33052632],
       [0.05894737],
       [0.05473684],
       [0.15789474]])

Verifying the original Values using inverse Scaling

In [86]:
rev = scaler.inverse_transform(energy_scaled)

In [90]:
rev[:10]

array([[1.  ],
       [1.09],
       [0.3 ],
       [0.5 ],
       [1.12],
       [1.68],
       [0.68],
       [1.79],
       [4.42],
       [1.52]])

In [103]:
def create_sequences( data, time_steps):
    X, Y = [], []
    for i in range(time_steps, len(data)):
        X.append(data[i-time_steps:i, 0])
        Y.append(data[i, 0])
    return np.array(X), np.array(Y)

In [104]:
TIME_STEPS = 24
X, Y = create_sequences(energy_scaled, TIME_STEPS)

Verifying the Results

In [109]:
X[:2]

array([[0.18947368, 0.20842105, 0.04210526, 0.08421053, 0.21473684,
        0.33263158, 0.12210526, 0.35578947, 0.90947368, 0.29894737,
        0.35789474, 0.38526316, 0.28631579, 0.36421053, 0.28      ,
        0.39578947, 0.11368421, 0.31578947, 0.89894737, 0.04631579,
        0.15578947, 0.38105263, 0.01473684, 0.05473684],
       [0.20842105, 0.04210526, 0.08421053, 0.21473684, 0.33263158,
        0.12210526, 0.35578947, 0.90947368, 0.29894737, 0.35789474,
        0.38526316, 0.28631579, 0.36421053, 0.28      , 0.39578947,
        0.11368421, 0.31578947, 0.89894737, 0.04631579, 0.15578947,
        0.38105263, 0.01473684, 0.05473684, 0.66105263]])

In [110]:
Y[:2]

array([0.66105263, 0.02315789])

Spliting in Train, Test

In [111]:
split_index = int(0.8 * len(Y))
X_train, X_test = X[:split_index], X[split_index:]
Y_train, Y_test = Y[:split_index], Y[split_index:]

In [122]:
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

In [123]:
model = Sequential()
model.add(LSTM(64, return_sequences=True, input_shape=(TIME_STEPS, 1)))
model.add(Dropout(0.2))
model.add(LSTM(32))
model.add(Dropout(0.2))
model.add(Dense(1))

c:\Python\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [125]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mean_squared_error'
)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    X_train,
    Y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
2229/2229 ━━━━━━━━━━━━━━━━━━━━ 47s 19ms/step - loss: 0.0576 - val_loss: 0.0568
Epoch 2/30
2229/2229 ━━━━━━━━━━━━━━━━━━━━ 43s 19ms/step - loss: 0.0576 - val_loss: 0.0568
Epoch 3/30
2229/2229 ━━━━━━━━━━━━━━━━━━━━ 43s 19ms/step - loss: 0.0576 - val_loss: 0.0568
Epoch 4/30
2229/2229 ━━━━━━━━━━━━━━━━━━━━ 43s 19ms/step - loss: 0.0576 - val_loss: 0.0568
Epoch 5/30
2229/2229 ━━━━━━━━━━━━━━━━━━━━ 52s 23ms/step - loss: 0.0576 - val_loss: 0.0568
Epoch 6/30
2229/2229 ━━━━━━━━━━━━━━━━━━━━ 44s 20ms/step - loss: 0.0576 - val_loss: 0.0568


In [126]:
y_pred_scaled = model.predict(X_test)

619/619 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step


In [127]:
y_test_actual = scaler.inverse_transform(Y_test.reshape(-1, 1))
y_pred_actual = scaler.inverse_transform(y_pred_scaled)

In [128]:
mae = mean_absolute_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
print("LSTM MAE:", mae)
print("LSTM RMSE:", rmse)

LSTM MAE: 0.8602533131004493
LSTM RMSE: 1.1326861788047624
